In [1]:
import os
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
tqdm.pandas()

# Retrieved from: https://amazon-reviews-2023.github.io/
# NOTE: Use 2023 dataset as it is larger, more descriptive, more granular and cleaner compared to
#       previous datasets.
dataset_name = "McAuley-Lab/Amazon-Reviews-2023"

# NOTE: There are 33 categories + Unknown. Software is a subset.
category = "Software"

d:\AI\LLM\projects\Recommender\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load User Reviews

In [2]:
#######################################################################################################
# Data Fields for User Reviews
#######################################################################################################
#
# - rating (float)             -->   Rating of the product (from 1.0 to 5.0)
# - title (str)                -->   Title of the user review
# - text (str)                 -->   Text body of the user review
# - images (list)              -->   Images that users post after they have received the product
#                                    Each image has different sizes (small, medium, large), represented
#                                    by the small_image_url, medium_image_url, and large_image_url
#                                    respectively
# - asin (str)                 -->   ID of the product
# - parent_asin (str)          -->   Parent ID of the product. Note: Products with different colors,
#                                    styles, sizes usually belong to the same parent ID. The “asin”
#                                    in previous Amazon datasets is actually parent ID. Please use
#                                    parent ID to find product meta
# - user_id (str)              -->   ID of the reviewer
# - timestamp (int)            -->   Time of the review (unix time)
# - verified_purchase (bool)   -->   User purchase verification
# - helpful_vote (int)         -->   Helpful votes of the review
#
#######################################################################################################

review_dataset = load_dataset(dataset_name, f"raw_review_{category}", trust_remote_code=True)

all_reviews = [review for review in tqdm(review_dataset["full"])]
review_df = pd.DataFrame(all_reviews)

review_df

Generating full split: 4880181 examples [01:06, 73630.22 examples/s]
100%|██████████| 4880181/4880181 [04:58<00:00, 16353.89it/s]


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1.0,malware,mcaffee IS malware,[],B07BFS3G7P,B0BQSK9QCF,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1562182632076,0,False
1,5.0,Lots of Fun,I love playing tapped out because it is fun to...,[],B00CTQ6SIG,B00CTQ6SIG,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,1424120336000,0,True
2,5.0,Light Up The Dark,I love this flashlight app! It really illumin...,[],B0066WJLU6,B0066WJLU6,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,1362399267000,0,True
3,4.0,Fun game,One of my favorite games,[],B00KCYMAWK,B00KCYMAWK,AH6CATODIVPVUOJEWHRSRCSKAOHA,1561061428662,0,True
4,4.0,I am not that good at it but my kids are,Cute game. I am not that good at it but my kid...,[],B00P1RK566,B00P1RK566,AEINY4XOINMMJCK5GZ3M6MMHBN6A,1418257196000,0,True
...,...,...,...,...,...,...,...,...,...,...
4880176,5.0,Gog,Very fun and addictive and exciting,[],B0763N3MVW,B0763N3MVW,AGOKECUMAI3W6MC7A7KBFBJVPN7Q,1608919468753,0,True
4880177,1.0,WORST GAME EVER,WORST GAME EVER TOXIC PEOPLE AND BAD CONNECTIO...,[],B00ET56Y48,B00ET56Y48,AH6AMUNDOSZ2SGDIDMYSRE4RNMDQ,1609973647689,1,True
4880178,5.0,better!!!,This fabulous game is 10000 times better than ...,[],B009ZKSPDK,B009ZKSPDK,AH4PJ73QN75AJM5VSCT53AOADCGA,1369722216000,2,True
4880179,5.0,It Has Everything I Need And More,Awesome! I upgraded from CorelDraw 8. I was wo...,[],B00M9J3IOA,B00M9J3IOA,AHQETDYKHVDNGMGBWVJL6VJXJGFQ,1472923291000,0,True


In [3]:
modified_review_df = review_df.copy()

modified_review_df["rating"] = pd.to_numeric(modified_review_df["rating"], errors="coerce")
modified_review_df["title"] = modified_review_df["title"].str.strip().str.lower()
modified_review_df["text"] = modified_review_df["text"].str.strip().str.lower()
modified_review_df["images"] = modified_review_df["images"].astype(str)
modified_review_df["asin"] = modified_review_df["asin"].str.lower()
modified_review_df["parent_asin"] = modified_review_df["parent_asin"].str.lower()
modified_review_df["user_id"] = modified_review_df["user_id"].str.lower()
modified_review_df["datetime"] = pd.to_datetime(modified_review_df["timestamp"], unit="ms").dt.date

# Reorder columns
modified_review_df = modified_review_df[[
    "asin",
    "parent_asin",
    "user_id",
    # "timestamp",
    "datetime",
    "title",
    "text",
    "images",
    "verified_purchase",
    "helpful_vote",
    "rating"
]]

# Rename columns
modified_review_df = modified_review_df.rename(columns={
    "datetime": "review_datetime",
    "title": "review_title",
    "text": "review_text",
    "images": "review_images",
    "rating": "review_rating"
})

modified_review_df

,asin,parent_asin,user_id,review_datetime,review_title,review_text,review_images,verified_purchase,helpful_vote,review_rating
0,b07bfs3g7p,b0bqsk9qcf,agci7fah4gl5fi65hylkwtmfz2cq,2019-07-03,malware,mcaffee is malware,[],False,0,1.0
1,b00ctq6sig,b00ctq6sig,ahspldnw5oouk2plh7gxlacfbznq,2015-02-16,lots of fun,i love playing tapped out because it is fun to...,[],True,0,5.0
2,b0066wjlu6,b0066wjlu6,ahspldnw5oouk2plh7gxlacfbznq,2013-03-04,light up the dark,i love this flashlight app! it really illumin...,[],True,0,5.0
3,b00kcymawk,b00kcymawk,ah6catodivpvuojewhrsrcskaoha,2019-06-20,fun game,one of my favorite games,[],True,0,4.0
4,b00p1rk566,b00p1rk566,aeiny4xoinmmjck5gz3m6mmhbn6a,2014-12-11,i am not that good at it but my kids are,cute game. i am not that good at it but my kid...,[],True,0,4.0
...,...,...,...,...,...,...,...,...,...,...
4880176,b0763n3mvw,b0763n3mvw,agokecumai3w6mc7a7kbfbjvpn7q,2020-12-25,gog,very fun and addictive and exciting,[],True,0,5.0
4880177,b00et56y48,b00et56y48,ah6amundosz2sgdidmysre4rnmdq,2021-01-06,worst game ever,worst game ever toxic people and bad connectio...,[],True,1,1.0
4880178,b009zkspdk,b009zkspdk,ah4pj73qn75ajm5vsct53aoadcga,2013-05-28,better!!!,this fabulous game is 10000 times better than ...,[],True,2,5.0
4880179,b00m9j3ioa,b00m9j3ioa,ahqetdykhvdngmgbwvjl6vjxjgfq,2016-09-03,it has everything i need and more,awesome! i upgraded from coreldraw 8. i was wo...,[],True,0,5.0


### Load Item Metadata

In [4]:
#######################################################################################################
# Data Fields for Item Metadata
#######################################################################################################
#
# - main_category (str)	       -->   Main category (i.e., domain) of the product
# - title (str)	               -->   Name of the product
# - average_rating (float)     -->   Rating of the product shown on the product page
# - rating_number (int)        -->   Number of ratings in the product
# - features (list)            -->   Bullet-point format features of the product
# - description (list)         -->   Description of the product
# - price (float)              -->   Price in US dollars (at time of crawling)
# - images (list)              -->   Images of the product. Each image has different sizes (thumb, large,
#                                    hi_res). The “variant” field shows the position of image
# - videos (list)              -->   Videos of the product including title and url
# - store (str)	               -->   Store name of the product
# - categories (list)	       -->   Hierarchical categories of the product
# - details (dict)             -->   Product details, including materials, brand, sizes, etc
# - parent_asin (str)	       -->   Parent ID of the product
# - bought_together (list)     -->   Recommended bundles from the websites
#
#######################################################################################################

meta_dataset = load_dataset(dataset_name,f"raw_meta_{category}", split="full", trust_remote_code=True)

meta_df = pd.DataFrame(meta_dataset)

meta_df

Generating full split: 89251 examples [00:11, 7680.87 examples/s]


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Appstore for Android,Accupressure Guide,3.6,NaN,[All the pressing point has been explained wit...,[Acupressure technique is very ancient and ver...,0.0,"{'hi_res': [None, None, None, None], 'large': ...","{'title': [''], 'url': [''], 'user_id': ['']}",mAppsguru,[],"{""Release Date"": ""2015"", ""Date first listed on...",B00VRPSGEO,None,None,None
1,Appstore for Android,Ankylosaurus Fights Back - Smithsonian's Prehi...,4.0,NaN,[ENCOURAGE literacy skills with highlighted na...,[Join Ankylosaurus in this interactive book ap...,2.99,"{'hi_res': [None, None, None, None, None], 'la...","{'title': [''], 'url': [''], 'user_id': ['']}","Oceanhouse Media, Inc",[],"{""Release Date"": ""2014"", ""Date first listed on...",B00NWQXXHQ,None,None,None
2,Appstore for Android,Mahjong 2015,3.1,NaN,[Mahjong 2015 is a free solitaire matching gam...,[Mahjong 2015 is a free solitaire matching gam...,0.0,"{'hi_res': [None, None, None], 'large': ['http...","{'title': [''], 'url': [''], 'user_id': ['']}",sophiathach,[],"{""Release Date"": ""2014"", ""Date first listed on...",B00RFKP6AC,None,None,None
3,Appstore for Android,Jewels Brick Breakout,4.2,NaN,"[Game Features:, - Intuitive touch controls wi...",[Jewels Brick Breakout is a glowing jewels bri...,0.0,"{'hi_res': [None, None, None, None, None, None...","{'title': [''], 'url': [''], 'user_id': ['']}",Bad Chicken,[],"{""Release Date"": ""2015"", ""Date first listed on...",B00SP2QU0E,None,None,None
4,Appstore for Android,Traffic Police: Off-Road Cub,3.3,NaN,"[In this game you will find:, - Killer police ...",[Become the best road police officer in Cube C...,0.0,"{'hi_res': [None, None, None, None], 'large': ...","{'title': [''], 'url': [''], 'user_id': ['']}",Dast 2 For Metro,[],"{""Release Date"": ""2016"", ""Date first listed on...",B01DZIT64O,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89246,Software,How to Start a Commercial Loan Broker Plus Bus...,3.2,2.0,[Everything You Need to Know About Starting a ...,[The How to Start a Commercial Loan Brokerage ...,29.95,"{'hi_res': [None, None, None, None, None], 'la...","{'title': [], 'url': [], 'user_id': []}",HowToStartABusinessDB,"[Software, Business & Office, Business & Marke...","{""Is Discontinued By Manufacturer"": ""No"", ""Ite...",B006KWVRGI,None,None,None
89247,Software,Norton Uninstall Deluxe,3.7,2.0,[Safely removes programs and files you no long...,[The safe uninstaller from the makers of Norto...,None,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Symantec,"[Software, Utilities]","{""Is Discontinued By Manufacturer"": ""No"", ""Pac...",B000J54PRA,None,None,None
89248,None,Beginning Pro Tools LE 8 [Instant Access],3.0,3.0,[],[This video takes you from start to finish on ...,15.95,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Alfred Music,"[Software, Education & Reference]","{""Language"": ""English"", ""Item model number"": ""...",B00MG0OHB0,None,None,None
89249,None,Tell Me More Spanish Performance Version 9 (2 ...,1.5,2.0,[Spanish language-learning software with 2 lev...,"[From the Manufacturer, This award-winning lan...",209.0,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Auralog,"[Software, Education & Reference, Languages]","{""Language"": ""English, French"", ""Package Dimen...",B001GN6YUK,None,None,None


In [5]:
category_replace_dict = {
    "accounting": "accounting & finance",
    "antivirus": "antivirus & security",
    "education": "education & reference",
    "free one-day shipping i software": "free one-day shipping for software",
    "free one-day shipping on select software with your citi card": "free one-day shipping for software",
    "medicine": "medicine & health sciences",
    "photography": "photography & graphic design",
    "programming": "programming & web development",
    "spreadsheet": "spreadsheet & database",
    "training": "training & tutorials"
}

def parse_categories(cat_str):
    """Parse string representation of list into actual list"""
    if pd.isna(cat_str) or cat_str.strip() == "[]":
        return []
    try:
        parsed_list = ast.literal_eval(cat_str)

        modified_list = []

        for elem in parsed_list:
            modified_elem = (
                elem
                .lower()
                .replace("\'s", "")
            )

            if modified_elem in category_replace_dict.keys():
                modified_list.append(category_replace_dict[modified_elem])
            else:
                modified_list.append(modified_elem)

        return sorted(modified_list)

    except Exception as e:
        print(f"{e}: {cat_str}")
        return [cat_str]

def parse_videos(video_dict):
    for key, val in video_dict.items():
        if val == [""]:
            video_dict[key] = []

    return video_dict

modified_meta_df = meta_df.copy()

modified_meta_df["parent_asin"] = modified_meta_df["parent_asin"].str.lower()
modified_meta_df["title"] = modified_meta_df["title"].str.strip().str.lower()
modified_meta_df["main_category"] = modified_meta_df["main_category"].str.strip().str.lower()
modified_meta_df["categories"] = modified_meta_df["categories"].astype(str).apply(parse_categories)
modified_meta_df["videos"] = modified_meta_df["videos"].apply(parse_videos)

# modified_meta_df["average_rating"] = pd.to_numeric(modified_meta_df["average_rating"], errors="coerce")
modified_meta_df["rating_number"] = pd.to_numeric(modified_meta_df["rating_number"], errors="coerce")

modified_meta_df["store"] = modified_meta_df["store"].astype(str).str.strip().str.lower()
modified_meta_df["price"] = pd.to_numeric(modified_meta_df["price"], errors="coerce")

# Reorder columns
modified_meta_df = modified_meta_df[[
    "parent_asin",
    "title",
    "main_category",
    "categories",
    "description",
    "features",
    "details",
    "images",
    "videos",
    # "bought_together", # All rows contain None
    # "average_rating", # Removing since more concern about rating on platform than product page
    "rating_number",
    "store",
    # "subtitle", # All rows contain None
    # "author", # All rows contain None
    "price"
]]

# Rename columns
modified_meta_df = modified_meta_df.rename(columns={
    "title": "item_title",
    "images": "item_images",
    "videos": "item_videos",
    "rating_number": "item_rating"
})

modified_meta_df

name 'ast' is not defined: ['Software', 'Music', 'Instrument Instruction']
name 'ast' is not defined: ['Software', "Children's", 'Geography']
name 'ast' is not defined: ['Software', 'Corel', 'All Corel']
name 'ast' is not defined: ['Software', 'Business & Office', 'Spreadsheet']
name 'ast' is not defined: ['Software', 'Business & Office', 'Business & Marketing Plans', 'Business Planning']
name 'ast' is not defined: ['Software', 'Intuit', 'All Intuit']
name 'ast' is not defined: ['Software', 'Microsoft', 'All Microsoft']
name 'ast' is not defined: ['Software', 'Photography & Graphic Design', 'CAD & Graphic Design']
name 'ast' is not defined: ['Software', 'Photography & Graphic Design', 'Animation & Anime']
name 'ast' is not defined: ['Software', 'Antivirus & Security', 'Antivirus']
name 'ast' is not defined: ['Software', 'Lifestyle & Hobbies', 'Home Publishing', 'Clip Art']
name 'ast' is not defined: ['Software', 'Business & Office', 'Note Taking']
name 'ast' is not defined: ['Software'

,parent_asin,item_title,main_category,categories,description,features,details,item_images,item_videos,item_rating,store,price
0,b00vrpsgeo,accupressure guide,appstore for android,[],[Acupressure technique is very ancient and ver...,[All the pressing point has been explained wit...,"{""Release Date"": ""2015"", ""Date first listed on...","{'hi_res': [None, None, None, None], 'large': ...","{'title': [], 'url': [], 'user_id': []}",NaN,mappsguru,0.00
1,b00nwqxxhq,ankylosaurus fights back - smithsonian's prehi...,appstore for android,[],[Join Ankylosaurus in this interactive book ap...,[ENCOURAGE literacy skills with highlighted na...,"{""Release Date"": ""2014"", ""Date first listed on...","{'hi_res': [None, None, None, None, None], 'la...","{'title': [], 'url': [], 'user_id': []}",NaN,"oceanhouse media, inc",2.99
2,b00rfkp6ac,mahjong 2015,appstore for android,[],[Mahjong 2015 is a free solitaire matching gam...,[Mahjong 2015 is a free solitaire matching gam...,"{""Release Date"": ""2014"", ""Date first listed on...","{'hi_res': [None, None, None], 'large': ['http...","{'title': [], 'url': [], 'user_id': []}",NaN,sophiathach,0.00
3,b00sp2qu0e,jewels brick breakout,appstore for android,[],[Jewels Brick Breakout is a glowing jewels bri...,"[Game Features:, - Intuitive touch controls wi...","{""Release Date"": ""2015"", ""Date first listed on...","{'hi_res': [None, None, None, None, None, None...","{'title': [], 'url': [], 'user_id': []}",NaN,bad chicken,0.00
4,b01dzit64o,traffic police: off-road cub,appstore for android,[],[Become the best road police officer in Cube C...,"[In this game you will find:, - Killer police ...","{""Release Date"": ""2016"", ""Date first listed on...","{'hi_res': [None, None, None, None], 'large': ...","{'title': [], 'url': [], 'user_id': []}",NaN,dast 2 for metro,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...
89246,b006kwvrgi,how to start a commercial loan broker plus bus...,software,"[['Software', 'Business & Office', 'Business &...",[The How to Start a Commercial Loan Brokerage ...,[Everything You Need to Know About Starting a ...,"{""Is Discontinued By Manufacturer"": ""No"", ""Ite...","{'hi_res': [None, None, None, None, None], 'la...","{'title': [], 'url': [], 'user_id': []}",2.0,howtostartabusinessdb,29.95
89247,b000j54pra,norton uninstall deluxe,software,"[['Software', 'Utilities']]",[The safe uninstaller from the makers of Norto...,[Safely removes programs and files you no long...,"{""Is Discontinued By Manufacturer"": ""No"", ""Pac...","{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",2.0,symantec,NaN
89248,b00mg0ohb0,beginning pro tools le 8 [instant access],None,"[['Software', 'Education & Reference']]",[This video takes you from start to finish on ...,[],"{""Language"": ""English"", ""Item model number"": ""...","{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",3.0,alfred music,15.95
89249,b001gn6yuk,tell me more spanish performance version 9 (2 ...,None,"[['Software', 'Education & Reference', 'Langua...","[From the Manufacturer, This award-winning lan...",[Spanish language-learning software with 2 lev...,"{""Language"": ""English, French"", ""Package Dimen...","{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",2.0,auralog,209.00


### Export Data

In [6]:
OUTPUT_DIR = "../data/output"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Export DataFrame
modified_review_df.to_parquet(f"{OUTPUT_DIR}/review_data.parquet", index=False)
modified_meta_df.to_parquet(f"{OUTPUT_DIR}/meta_data.parquet", index=False)

print("Complete")

Complete
